# RAG Bench — 60-Combo Benchmark on Google Colab

한국어 RAG 파이프라인 60개 조합을 Google Colab T4 GPU에서 벤치마크합니다.

## 3-Layer Architecture
```
Layer 1: Dense Model ─── kosimcse | e5 | bge-m3                   (HuggingFace, 3종)
                         openai-large                              (OpenAI API, 1종)
                         upstage                                    (Upstage API, 1종)
                                                                    총 5종
Layer 2: Sparse Model ── korean_bm25 | splade                    (2종)
Layer 3: Retrieval Mode ─ hybrid × reranker × llm_support        (6종)
                          ├── hybrid (기본)
                          ├── hybrid + contextual
                          ├── hybrid + colbert_rerank
                          ├── hybrid + colbert_rerank + contextual
                          ├── hybrid + flashrank_rerank
                          └── hybrid + flashrank_rerank + contextual

총 조합: 5 × 2 × 6 = 60개
```

## 전체 실행 흐름
```
[Section 1] init_colab()         환경 초기화 (Drive, API 키, 패치)
[Section 2] 사용자 설정           PRESET / K / TOP_N / QDRANT_MODE
[Section 3] runner.prepare_qa()  PDF 샘플링 → RAGAS KG → QA 데이터셋
[Section 4] runner.prepare_data() QA 로드 + Parent-Child 청킹
[Section 5] runner.generate_combos() 벤치마크 조합 생성
[Section 6] runner.run_pass1()   전체 조합 레이턴시 측정
[Section 7] runner.run_pass2()   상위 전략 RAGAS 평가
[Section 8] 시각화 대시보드
[Section 9] 결과 저장 + HTML 보고서
```

## 예상 실행시간 (T4 GPU)
| 프리셋 | 조합 수 | Pass 1 | Pass 2 | 총 예상 |
|--------|---------|--------|--------|--------|
| quick | 4 | ~3분 | ~10분 | ~15분 |
| standard | 42 | ~40분 | ~60분 | ~1.5시간 |
| full | 126 | ~2시간 | ~3시간 | ~5시간 |

> **참고**: OpenAI/Upstage API 모델은 GPU 불필요, API 키 필요. HuggingFace 모델은 T4 GPU 사용.

---
## Section 1: 환경 설정

In [ ]:
# Cell 1.2: 시스템 의존성 + 패키지 설치 (flash-attn wheel 캐시 최적화)
import os
import sys
import subprocess
from pathlib import Path

# ── 프로젝트 경로 추가 ──
_project_root = Path("/content/RAG-Bench")
for _p in [str(_project_root), str(_project_root / "rag_bench_colab")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── 시스템 의존성 ──
os.system("apt-get update -qq")
os.system("apt-get install -y default-jdk -qq --fix-missing")

# ── Torch 버전 보존 (CUDA 호환성) ──
_torch_ver = subprocess.check_output(
    ["python", "-c", "import torch; print(torch.__version__)"]
).decode().strip()
print(f"[torch] Colab 사전 설치 버전: {_torch_ver} (보존)")

# ── Drive 마운트 ──
_drive_ok = Path("/content/drive/MyDrive").exists()
if not _drive_ok:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        _drive_ok = True
    except Exception:
        _drive_ok = False
_colab_cache = Path("/content/drive/MyDrive/colab_cache") if _drive_ok else None

# ── 패키지 설치 헬퍼 ──
def _pip(pkgs: str, label: str = "", upgrade: bool = False):
    flag = "--upgrade " if upgrade else ""
    ret = os.system(f"pip install --prefer-binary -q {flag}{pkgs}")
    print(f"[pip] {'✓' if ret == 0 else '⚠'} {label or pkgs}")
    return ret

# ── 1) qdrant-client ──
_pip("'qdrant-client>=1.11'", "qdrant-client")

# ── 2) LangChain 에코시스템 — --upgrade로 호환 버전 세트 일관 설치 ──
_pip(
    "'langchain>=1.0' 'langchain-core>=0.3' 'langchain-community>=0.3'"
    " 'langchain-text-splitters>=0.3' 'langchain-openai>=0.2'"
    " 'langchain-huggingface>=0.1' 'langchain-qdrant>=1.0'"
    " 'langchain-upstage>=0.3' 'langgraph>=0.2'",
    "langchain 에코시스템", upgrade=True,
)

# ── langchain sys.modules 캐시 무효화 ──
# pip upgrade가 파일을 교체해도 이미 로드된 모듈은 캐시에서 유지됨
# → 명시적으로 제거해야 다음 import 시 새 파일을 로드함
import importlib
importlib.invalidate_caches()
_cleared = [k for k in list(sys.modules) if "langchain" in k or "openai" in k]
for k in _cleared:
    del sys.modules[k]
print(f"[cache] langchain/openai 모듈 캐시 {len(_cleared)}개 초기화")

# ── langchain_core 버전 검증 (ContextOverflowError 존재 여부) ──
try:
    from langchain_core.exceptions import ContextOverflowError  # noqa: F401
    print("[check] langchain_core.ContextOverflowError ✓")
except ImportError:
    print("[check] ⚠ langchain_core 버전 불일치 → 강제 업그레이드")
    os.system("pip install --upgrade 'langchain-core>=0.3.28' 'langchain-openai>=0.2'")
    importlib.invalidate_caches()
    for k in [k for k in list(sys.modules) if "langchain" in k or "openai" in k]:
        del sys.modules[k]
    print("[check] 재설치 완료 — 런타임 재시작 없이 적용됨")

# ── 3) ML / NLP ──
_pip(
    "'sentence-transformers>=3.1' 'transformers>=4.40'"
    " 'pylate>=1.0' 'flashrank>=0.2' 'einops>=0.8.2'"
    " 'fastembed>=0.4' 'konlpy>=0.6.0'",
    "ML/NLP"
)

# ── 4) PDF ──
_pip("'pymupdf>=1.24' 'pymupdf4llm>=0.0.17'", "PDF")

# ── 5) Evaluation ──
_pip("'ragas>=0.4.3' 'rapidfuzz>=3.0'", "ragas")

# ── 6) Utils ──
_pip("'python-dotenv>=1.0' 'pandas>=2.2' 'httpx'", "utils")

# ── 7) Visualization (실패해도 계속) ──
os.system(
    "pip install --prefer-binary -q"
    " 'matplotlib>=3.10' 'plotly>=5.18' 'seaborn>=0.13'"
    " 'kaleido>=0.2' 'koreanize-matplotlib>=0.1'"
    " 2>/dev/null; true"
)
print("[pip] ✓ viz")

# ── flash-attn: 사전 빌드 wheel만 설치 ──
import torch

if not torch.cuda.is_available():
    print("[flash-attn] CUDA 미감지 → 건너뜀")
else:
    FA_VER = "2.8.3"
    _FA_WHEEL_TORCH = ["2.4", "2.5", "2.6", "2.7", "2.8", "2.9"]
    _torch_xy = ".".join(_torch_ver.split(".")[:2])
    _cu_str   = "12" if "+cu12" in _torch_ver else "11"
    _py_ver   = f"{sys.version_info.major}{sys.version_info.minor}"
    try:
        _cxx11 = "TRUE" if torch.compiled_with_cxx11_abi() else "FALSE"
    except AttributeError:
        _cxx11 = "FALSE"

    if _torch_xy not in _FA_WHEEL_TORCH:
        print(f"[flash-attn] torch {_torch_xy} — FA {FA_VER} 미지원 → 건너뜀")
    else:
        _wheel_name = (
            f"flash_attn-{FA_VER}+cu{_cu_str}torch{_torch_xy}"
            f"cxx11abi{_cxx11}-cp{_py_ver}-cp{_py_ver}-linux_x86_64.whl"
        )
        _wheel_url  = (
            f"https://github.com/Dao-AILab/flash-attention/releases/download"
            f"/v{FA_VER}/{_wheel_name}"
        )
        print(f"[flash-attn] 대상 wheel: {_wheel_name}")
        _drive_wheels = _colab_cache / "wheels" if _colab_cache else None
        _cached_wheel = _drive_wheels / _wheel_name if _drive_wheels else None
        _installed    = False

        if _cached_wheel and _cached_wheel.exists():
            print(f"[flash-attn] Drive 캐시 사용 → {_cached_wheel.name}")
            _installed = os.system(f"pip install -q {_cached_wheel}") == 0

        if not _installed:
            print("[flash-attn] GitHub wheel 다운로드 중...")
            _tmp = Path(f"/tmp/{_wheel_name}")
            if os.system(f"wget -q --show-progress -O {_tmp} {_wheel_url}") == 0 \
                    and _tmp.exists() and _tmp.stat().st_size > 1_000_000:
                _installed = os.system(f"pip install -q {_tmp}") == 0
                if _installed and _drive_wheels:
                    try:
                        _drive_wheels.mkdir(parents=True, exist_ok=True)
                        import shutil; shutil.copy2(_tmp, _cached_wheel)
                        print(f"[flash-attn] Drive 캐시 저장 → {_cached_wheel.name}")
                    except Exception as _e:
                        print(f"[flash-attn] Drive 캐시 저장 실패: {_e}")
            else:
                print("[flash-attn] 다운로드 실패 → 건너뜀")

        print(f"[flash-attn] {'✓ 설치 완료' if _installed else 'wheel 없음 — 건너뜀'}")

print("\n[완료] 패키지 설치 완료")
print(f"[sys.path] {sys.path[:3]} ...")

In [ ]:
# Cell 1.3: Colab 환경 초기화 + rag_bench 패치 + smoke test
import sys
import os
from pathlib import Path

# ── 경로 보장 (Cell 1.2를 건너뛴 경우 대비) ──
for _p in ["/content/RAG-Bench", "/content/RAG-Bench/rag_bench_colab"]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── sys.modules 캐시 초기화 (이전 실패한 import 잔재 제거) ──
sys.modules.pop("colab_config", None)

# ── 진단: colab_config.py 실제 존재 여부 확인 ──
_cfg_path = Path("/content/RAG-Bench/rag_bench_colab/colab_config.py")
if not _cfg_path.exists():
    print(f"[ERROR] {_cfg_path} 파일이 없습니다!")
    print("[진단] /content/RAG-Bench 디렉토리:")
    os.system("ls /content/RAG-Bench/ 2>&1 | head -20")
    raise FileNotFoundError(f"colab_config.py not found at {_cfg_path}")

# ── importlib로 직접 로드 (sys.path 우선순위 문제 우회) ──
import importlib.util as _ilu

_spec = _ilu.spec_from_file_location("colab_config", str(_cfg_path))
_mod  = _ilu.module_from_spec(_spec)
sys.modules["colab_config"] = _mod
_spec.loader.exec_module(_mod)

init_colab = _mod.init_colab
print(f"[colab_config] ✓ {_cfg_path} 직접 로드 완료")

env_info = init_colab(
    qdrant_mode="ephemeral",  # 'ephemeral' | 'drive' | 'memory'
    device=None,               # None = 자동 감지
    mount_drive=True,
)

# ── Smoke test: 핵심 모듈 import (비치명적) ──
_smoke_ok = True
_smoke_missing = []

for _mod_name, _import_str in [
    ("rag_bench.config",  "from rag_bench.config import BENCH_DATA_DIR, BENCH_DOCS_DIR"),
    ("rag_bench.combo",   "from rag_bench.combo import PRESETS, generate_valid_combinations, ComboSpec"),
    ("rag_bench.runner",  "from rag_bench.runner import BenchmarkRunner"),
]:
    try:
        exec(_import_str)
    except ImportError as e:
        _smoke_ok = False
        _smoke_missing.append(f"  ✗ {_mod_name}: {e}")

if _smoke_ok:
    print("\n[Smoke Test] ✓ 모든 import 성공!")
    print(f"  BENCH_DATA_DIR: {BENCH_DATA_DIR}")
    print(f"  BENCH_DOCS_DIR: {BENCH_DOCS_DIR}")
else:
    print("\n[Smoke Test] ⚠ 일부 import 실패 — Cell 1.2 재실행 후 Cell 1.3 재실행 필요:")
    for _m in _smoke_missing:
        print(_m)
    print("\n  → Colab 메뉴: 런타임 → 런타임 다시 시작 후 Cell 1.1부터 재실행")

In [4]:
# Cell 1.4: API Key 설정 (자동 로드 실패 시 수동 입력)
import os

if not env_info.get("api_key_loaded", False):
    import getpass
    _key = getpass.getpass("OPENAI_API_KEY 입력 (Enter로 건너뜀): ")
    if _key.strip():
        os.environ["OPENAI_API_KEY"] = _key.strip()
        print("[API Key] 수동 입력 완료")
    else:
        print("[Warning] OPENAI_API_KEY 미설정 — Pass 2 RAGAS 평가가 실패할 수 있습니다.")
else:
    print("[API Key] 이미 로드됨")

In [5]:
# Cell 1.5: Upstage API Key 설정 (자동 로드 실패 시 수동 입력)
import os

if not env_info.get("upstage_api_key_loaded", False):
    import getpass
    _key = getpass.getpass("UPSTAGE_API_KEY 입력 (Enter로 건너뜀): ")
    if _key.strip():
        os.environ["UPSTAGE_API_KEY"] = _key.strip()
        print("[Upstage API Key] 수동 입력 완료")
    else:
        print("[Warning] UPSTAGE_API_KEY 미설정 — UpstageEmbedStrategy 사용 시 필요합니다.")
else:
    print("[Upstage API Key] 이미 로드됨")

---
## Section 2: 사용자 설정

### QDRANT_MODE 선택 가이드

| 모드 | 저장 위치 | 세션 종료 후 | 추천 상황 |
|------|-----------|-------------|-----------|
| `ephemeral` | `/content/qdrant_workspace` (로컬) | **삭제됨** | 빠른 테스트, 재현 불필요 |
| `drive` | `Google Drive/MyDrive/rag_bench_colab/` | **유지됨** | 장시간 실험, 결과 보존 필요 |
| `memory` | 메모리 (RAM) | **삭제됨** | 가장 빠름, 소규모 실험 |

> **권장**: 처음 실행은 `ephemeral`, 결과를 보존하려면 `drive`

### 주요 파라미터

| 파라미터 | 설명 |
|----------|------|
| `PRESET` | 벤치마크 조합 수 — `quick` (4) / `standard` (24) / `full` (72) |
| `K` | 검색 시 반환할 문서 수 |
| `TOP_N` | Pass 1 완료 후 RAGAS 평가할 상위 전략 수 |
| `METRIC_PRESET` | 평가 메트릭 — `core_only` (4) / `comprehensive` (7) / `full` (11+) / `reference_free` |
| `SCORING_PROFILE` | 가중 점수 프로파일 — `balanced` / `precision_critical` / `speed_critical` / `comprehensive` |

In [6]:
# ===== 사용자 설정 =====
PRESET = "full"     # 'quick' (4조합) | 'standard' (24) | 'full' (72)
K = 3                # 검색 결과 수
TOP_N = 6            # Pass 2 RAGAS 평가 대상 (상위 N)

# QDRANT_MODE:
#   'ephemeral' → /content/qdrant_workspace  (세션 종료 시 삭제, 기본값)
#   'drive'     → Google Drive/rag_bench_colab/ (영구 보존, mount_drive=True 필요)
#   'memory'    → RAM 전용 (가장 빠름, 세션 종료 시 삭제)
QDRANT_MODE = "drive"

# 평가 설정 (rag_bench evaluation 최신화 반영)
METRIC_PRESET = "full"    # 'core_only' (4) | 'comprehensive' (7) | 'full' (11+) | 'reference_free'
SCORING_PROFILE = "balanced"

# Pass 2 RAGAS 병렬 평가 (0=비활성, T4에서 4~8 권장)
PARALLEL_EVAL = 4   # 'balanced' | 'precision_critical' | 'speed_critical' | 'comprehensive'
# ======================

print(f"Preset: {PRESET}")
print(f"K: {K}, Top-N: {TOP_N}")
print(f"Qdrant Mode: {QDRANT_MODE}")
print(f"Metric Preset: {METRIC_PRESET}")
print(f"Scoring Profile: {SCORING_PROFILE}")

In [ ]:
# [선택] Drive 데이터 초기화 — 데이터 오염 방지
# 이전 실행의 캐시가 새 실행에 영향을 줄 수 있을 때 실행하세요.
# 각 플래그를 True로 설정 후 이 셀만 실행합니다.
import glob, shutil, time
from pathlib import Path

# ──────────── 초기화 대상 선택 ────────────────────────────────────────────────
CLEAN_CHECKPOINTS = False  # 이전 세션 체크포인트 삭제 (조합 구조가 바뀐 경우)
CLEAN_QA_CACHE    = False  # qa_dataset.json + RAGAS KG + parent_store 삭제
                           # (새 문서로 벤치마크하거나 샘플링 비율 변경 시)
CLEAN_QDRANT_DATA = False  # Qdrant drive 모드 인덱스 삭제 (drive 모드 사용 시만)
CLEAN_RESULTS     = False  # 결과 CSV/HTML 삭제 (기본 유지 권장)
# ─────────────────────────────────────────────────────────────────────────────

_DRIVE = Path("/content/drive/MyDrive/rag_bench_colab")

def _du(p: Path) -> str:
    "경로 크기·수정일 문자열."
    if not p.exists():
        return "(없음)"
    mtime = time.strftime("%m-%d %H:%M", time.localtime(p.stat().st_mtime))
    if p.is_dir():
        size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        count = sum(1 for _ in p.rglob("*.json"))
        return f"{size/1024/1024:.1f}MB  {mtime}  [.json {count}개]"
    return f"{p.stat().st_size/1024:.1f}KB  {mtime}"

print("═" * 58)
print("  Drive 데이터 현황  (MyDrive/rag_bench_colab/)")
print("═" * 58)
_items = [
    ("checkpoints/",          _DRIVE / "checkpoints"),
    ("_benchdata/qa_dataset.json",  _DRIVE / "_benchdata" / "qa_dataset.json"),
    ("_benchdata/ragas_kg.json",    _DRIVE / "_benchdata" / "ragas_knowledge_graph.json"),
    ("_benchdata/parent_store/",    _DRIVE / "_benchdata" / "parent_store"),
    ("results/",              _DRIVE / "results"),
]
for label, p in _items:
    print(f"  {label:<38} {_du(p)}")

# Qdrant drive 인덱스 (있을 경우)
_qdrant_dirs = glob.glob(str(_DRIVE / "_benchdata" / "qdrant_db_*"))
if _qdrant_dirs:
    for _qd in _qdrant_dirs:
        p = Path(_qd)
        print(f"  _benchdata/{p.name:<28} {_du(p)}")

print()

# ─── 초기화 실행 ──────────────────────────────────────────────────────────────
_cleaned = []

if CLEAN_CHECKPOINTS:
    p = _DRIVE / "checkpoints"
    if p.exists():
        shutil.rmtree(p); p.mkdir(parents=True)
        _cleaned.append("checkpoints/")

if CLEAN_QA_CACHE:
    for fname in ["qa_dataset.json", "ragas_knowledge_graph.json"]:
        p = _DRIVE / "_benchdata" / fname
        if p.exists(): p.unlink(); _cleaned.append(fname)
    p = _DRIVE / "_benchdata" / "parent_store"
    if p.exists(): shutil.rmtree(p); _cleaned.append("parent_store/")

if CLEAN_QDRANT_DATA:
    for _qd in glob.glob(str(_DRIVE / "_benchdata" / "qdrant_db_*")):
        shutil.rmtree(_qd); _cleaned.append(Path(_qd).name)

if CLEAN_RESULTS:
    p = _DRIVE / "results"
    if p.exists(): shutil.rmtree(p); p.mkdir(parents=True); _cleaned.append("results/")

if _cleaned:
    print(f"[정리 완료] 삭제: {', '.join(_cleaned)}")
    print("  → Cell 3.2 prepare_qa(force=True) 또는 새 러너로 재시작하세요.")
else:
    print("[정리] 삭제 항목 없음")
    print("  → 필요한 플래그를 True로 바꾼 뒤 이 셀을 다시 실행하세요.")


---
## Section 3: QA 데이터셋 생성

PDF 원본에서 RAGAS 기반 QA 쌍을 자동 생성합니다.  
이미 `qa_dataset.json`이 존재하면 캐시를 사용하고 건너뜁니다 (`force=True`로 강제 재생성).

> **의존성**: `init_colab()`을 먼저 실행해야 Colab 경로 패치가 적용됩니다.

In [7]:
# Cell 3.1: 러너 생성
from colab_runner import ColabBenchmarkRunner

runner = ColabBenchmarkRunner(
    preset=PRESET,
    k=K,
    top_n=TOP_N,
    qdrant_mode=QDRANT_MODE,
    metric_preset=METRIC_PRESET,
    scoring_profile=SCORING_PROFILE,
    parallel_eval=PARALLEL_EVAL,
)
print(f"[Runner] preset={PRESET}, k={K}, top_n={TOP_N}, qdrant_mode={QDRANT_MODE}")

In [ ]:
# Cell 3.2: QA 데이터셋 생성 (PDF 페이지 샘플링)
# docs/*.pdf → 10% 샘플링 → data/docs/*.md 재변환 후 RAGAS KG 기반 QA 생성
# QA 수 = 청크 수 × max_qa_per_page (자동 결정)
# 이미 qa_dataset.json이 존재하면 캐시 사용 (force=True로 강제 재생성)
runner.prepare_qa(
    sample_pages=True,       # docs/*.pdf 페이지 샘플링 후 재변환
    page_sample_ratio=0.1,   # 샘플링 비율 (10%)
    max_sample_pages=5,      # 최대 샘플 페이지 수
    max_qa_per_page=2,       # 청크당 QA 수 → 총 QA = 청크 수 × 2
    force=True,              # True: 캐시 무시하고 강제 재생성
    query_dist="balanced",   # single_hop | multi_hop | balanced
)

---
## Section 4: 데이터 로딩

In [ ]:
# Cell 4.1: 데이터 로드 + 청킹
child_chunks, parent_pairs, queries, ground_truths = runner.prepare_data()

print(f"\nQA 샘플:")
for i, q in enumerate(queries[:3]):
    print(f"  Q{i+1}: {q[:80]}...")
    print(f"  A{i+1}: {ground_truths[i][:80]}...")

In [ ]:
# Cell 4.2: Parent-Child 청킹 통계
print(f"Parent 청크: {len(parent_pairs)}개")
print(f"Child 청크: {len(child_chunks)}개")
print(f"\n샘플 Child 청크 (첫 번째):")
print(child_chunks[0].page_content[:300])

---
## Section 5: 조합 생성

In [ ]:
# 프리셋 기반 ComboSpec 생성
combos = runner.generate_combos()

import pandas as pd
combo_table = pd.DataFrame([
    {
        "#": i+1,
        "Label": spec.label,
        "Dense": spec.dense,
        "Sparse": spec.sparse,
        "Reranker": spec.reranker or "-",
        "LLM Support": spec.llm_support or "-",
    }
    for i, spec in enumerate(combos)
])
display(combo_table)

In [ ]:
# Step 2.5: Contextual 청크 사전 생성 (최적화)
# contextual 조합이 있을 때만 enrich_only() 1회 호출 — API 비용 대폭 절감
pre_enriched = runner.prepare_contextual_chunks(combos, child_chunks, parent_pairs)

---
## Section 6: Pass 1 — 레이턴시 벤치마크

In [ ]:
# Pass 1: 전체 조합 레이턴시 측정
latency_df = runner.run_pass1(combos, queries, child_chunks, parent_pairs)
display(latency_df)

In [ ]:
# Pass 1 시각화
from colab_visualizer import plot_latency_comparison
plot_latency_comparison(latency_df)

---
## Section 7: Pass 2 — RAGAS 평가

In [ ]:
# Pass 2: 상위 N개 전략 RAGAS 평가 (체크포인트 지원)
ragas_df = runner.run_pass2(
    latency_df, combos, queries, ground_truths,
    child_chunks, parent_pairs,
)
display(ragas_df)

In [ ]:
# RAGAS 결과 스타일링 테이블
from colab_visualizer import display_styled_table, display_weighted_scores
display_styled_table(ragas_df)

# Weighted Score (프로파일별 가중 점수)
if runner.reports:
    print(f"\n--- Weighted Scores (profile={SCORING_PROFILE}) ---")
    display_weighted_scores(runner.reports, scoring_profile=SCORING_PROFILE)

---
## Section 8: 시각화 대시보드

수행 이력(RunTracker) + RAGAS 차트 + 가중 점수(Weighted Score) 통합 시각화

In [ ]:
# 수행 이력 요약 (RunTracker 연동)
run_record = runner.get_run_record()
if run_record:
    from colab_visualizer import plot_run_info, plot_phase_timeline
    print("--- Run Summary ---")
    plot_run_info(run_record)
    print("\n--- Phase Timeline ---")
    plot_phase_timeline(run_record)
else:
    print("RunTracker 데이터가 없습니다.")

In [ ]:
# 히트맵 (전략 x 메트릭)
from colab_visualizer import plot_ragas_heatmap
plot_ragas_heatmap(ragas_df)

In [ ]:
# 파레토 프론티어 (레이턴시 vs 품질)
from colab_visualizer import plot_latency_vs_quality
plot_latency_vs_quality(latency_df, ragas_df)

In [ ]:
# 레이어별 기여도 분석
from colab_visualizer import plot_layer_contribution
plot_layer_contribution(combos, latency_df, metric="avg_latency")

In [ ]:
# [H1] Ablation Waterfall — 레이어별 품질 기여도
from colab_visualizer import plot_ablation_waterfall
plot_ablation_waterfall(ragas_df)

In [ ]:
# [H3] Layer Interaction Heatmap — Dense × Sparse 조합 상호작용
from colab_visualizer import plot_layer_interaction_heatmap
plot_layer_interaction_heatmap(ragas_df)

In [ ]:
# [H4] Tradeoff Bubble Chart — 레이턴시 × 품질 × 비용
from colab_visualizer import plot_tradeoff_bubble
plot_tradeoff_bubble(latency_df, ragas_df, run_record=run_record)

In [ ]:
# [H-2] Metric Violin — 전략 그룹별 per-sample 메트릭 분포
from colab_visualizer import plot_metric_violin
if runner.reports:
    plot_metric_violin(runner.reports)
else:
    print("[Info] Pass 2 RAGAS 평가 결과가 필요합니다. (runner.reports 비어있음)")

In [ ]:
# [M-1] Pipeline Diagram — RAG 파이프라인 구조 다이어그램
from colab_visualizer import plot_pipeline_diagram
if combos:
    # 대표 조합: contextual+reranker 조합 우선, 없으면 첫 번째
    sample_spec = next(
        (s for s in combos if getattr(s, "reranker", None) and getattr(s, "llm_support", None)),
        combos[0],
    )
    plot_pipeline_diagram(spec=sample_spec)
else:
    print("[Info] combos가 필요합니다. Section 5를 먼저 실행하세요.")

In [ ]:
# [M-3] Strategy Gantt — 전략 빌드 타임라인
from colab_visualizer import plot_strategy_gantt
run_record = runner.get_run_record()
if run_record and run_record.get("strategy_timings"):
    plot_strategy_gantt(run_record)
else:
    print("[Info] RunTracker 데이터가 없습니다. Pass 1 실행 후 사용하세요.")

In [ ]:
# [M-4] Cost-Efficiency — 비용 대비 품질 효율 산점도
from colab_visualizer import plot_cost_efficiency
run_record = runner.get_run_record()
if run_record and ragas_df is not None and not ragas_df.empty:
    plot_cost_efficiency(run_record, ragas_df, latency_df)
else:
    print("[Info] run_record와 ragas_df가 모두 필요합니다. Pass 1 + Pass 2 실행 후 사용하세요.")

---
## Section 9: 비용 요약 + 결과 저장

In [ ]:
# 비용 요약 (추정)
n_ragas_strategies = len(ragas_df) if ragas_df is not None else 0
n_queries = len(queries)
est_ragas_cost = n_ragas_strategies * n_queries * 0.005  # ~$0.005/query/strategy

cost_data = {
    "RAGAS 평가 (GPT-4o-mini)": est_ragas_cost,
    "Answer 생성 (GPT-4o-mini)": n_ragas_strategies * n_queries * 0.001,
}
total_cost = sum(cost_data.values())

print(f"예상 API 비용:")
for k, v in cost_data.items():
    print(f"  {k}: ${v:.2f}")
print(f"  총계: ${total_cost:.2f}")

from colab_visualizer import plot_cost_breakdown
if total_cost > 0:
    plot_cost_breakdown(cost_data)

In [ ]:
# 결과 Export (Google Drive)
output_dir = runner.export_results(
    latency_df=latency_df,
    ragas_df=ragas_df,
)
print(f"\n결과 저장 위치: {output_dir}")

In [ ]:
# HTML 보고서 열기
from IPython.display import HTML, display
html_path = output_dir / 'report.html'
if html_path.exists():
    display(HTML(f'<a href="{html_path}" target="_blank">📊 HTML 보고서 열기</a>'))
    print(f'HTML 보고서: {html_path}')
else:
    print('[Info] HTML 보고서가 아직 생성되지 않았습니다.')